# eCREST Reconstruction Workflow

This notebook documents the eCREST reconstruction workflow used to collect the ELL connectome data.

**eCREST** is a CLI-based Python interface built on top of [CREST](https://github.com/ashapsoncoe/CREST) (by Alex Shapson-Coe). It requires `neuroglancer` and a `settings_dict.json` file pointing to the agglomeration database. Refer to the [eCREST/CREST GitHub](https://github.com/ashapsoncoe/CREST) for full documentation on setup and configuration.

> **Note:** This notebook is for documentation purposes — the reconstruction data is already included in `EM_data_published/reconstructions_published/`. You do not need to re-run the reconstruction steps to reproduce the analysis.

## Section 1: Setup

In [1]:
import sys
from pathlib import Path

DIR_ROOT = Path.cwd().parent / 'efish_em'
if str(DIR_ROOT) not in sys.path:
    sys.path.insert(0, str(DIR_ROOT))

from eCREST_cli import ecrest
import AnalysisCode as efish
import json

In [5]:
# settings_dict is defined inline below (see cell 6); see README for details.
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'

vx_sizes = [16,16,30]

In [2]:
# settings_path = Path.cwd().parent / 'settings_dict.json'
# with open(settings_path) as f:
#     settings_dict = json.load(f)
# print("Settings loaded. Keys:", list(settings_dict.keys()))

Settings loaded. Keys: ['save_dir', 'max_num_base_added', 'cell_structures', 'annotation_points', 'db_path']


save_dir is set to the directory in which efish_em_ELL repository has been cloned and in which EM_data_published has been saved/downloaded.

In [3]:
settings_dict = {
    "save_dir": DATA_ROOT.parent,#"<default directory for reconstructions to save>",
    "max_num_base_added": 1000,
    "cell_structures": [
        "unknown",
        "axon",
        "basal dendrite",
        "apical dendrite",
        "dendrite",
        "multiple"
    ],
    "annotation_points": [
        "exit volume",
        "natural end",
        "uncertain",
        "pre-synaptic",
        "post-synaptic"
    ],
    "db_path": DATA_ROOT #"<path to agglomeration database file>/Mariela_bigquery_exports_agglo_v230111c_16_crest_proofreading_database.db"
}

{'save_dir': '<default directory for reconstructions to save>',
 'max_num_base_added': 1000,
 'cell_structures': ['unknown',
  'axon',
  'basal dendrite',
  'apical dendrite',
  'dendrite',
  'multiple'],
 'annotation_points': ['exit volume',
  'natural end',
  'uncertain',
  'pre-synaptic',
  'post-synaptic'],
 'db_path': '<path to agglomeration database file>/Mariela_bigquery_exports_agglo_v230111c_16_crest_proofreading_database.db'}

## Section 2: Load Reconstruction Files

The `.json` cell graph files in `reconstructions_published/` are the output of the eCREST reconstruction process. Each file contains the graph structure of a single neuron, including its base segments and synaptic endpoint annotations. These files were used as input for all downstream connectivity analyses.

In [ ]:
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'
nodefiles = efish.get_cell_filepaths(dirpath)
print(f"Found {len(nodefiles)} reconstruction files in: {dirpath}")

## Section 3: New Reconstruction from Segment ID

When a new segment (neuron) is identified in Neuroglancer, the `ecrest` class can be initialized with the root segment ID to begin reconstruction. This opens an interactive Neuroglancer viewer in which you can add or remove base segments, annotate synaptic endpoints, and save the cell graph.

Use this workflow when starting from a fresh segment ID rather than loading an existing file.

In [ ]:
# Initialize a new reconstruction from a Neuroglancer segment ID
# segment_id = 12345  # Replace with the actual segment ID from Neuroglancer
# crest = ecrest(settings_dict, segment_id=segment_id, launch_viewer=True)

In [ ]:
# Assign cell type after reconstruction
# cell_type = 'MG'  # e.g., 'MG', 'SG', 'GC', 'LG', 'LF'
# crest.define_ctype(cell_type, "manual")
# crest.save_cell_graph()

## Section 4: Edit Existing Reconstruction from File

To continue proofreading an existing reconstruction, load it by specifying its filepath from `nodefiles`. This opens the Neuroglancer viewer with the saved cell graph, allowing you to continue editing, add synaptic annotations, or correct the segment graph.

> You can also pass a full filepath directly to the `filepath` flag: `crest = ecrest(settings_dict, filepath=some_path, launch_viewer=True)`

> To change the save location, specify the `directory_path` flag in `save_cell_graph()`. To overwrite the original file (not recommended), use `directory_path=filepath.parent, file_name=filepath.name`.

In [ ]:
# Load an existing cell reconstruction by its index into nodefiles
# cell_id = 0  # index into nodefiles
# crest = ecrest(settings_dict, filepath=nodefiles[cell_id], launch_viewer=True)

In [ ]:
# Add annotation layers for proofreading synaptic endpoints
# crest.add_endpoint_annotation_layers(['soma'], link=True)

In [ ]:
# Check current cell type and update if needed
# print("Current cell type:", crest.get_ctype('manual'))
# crest.define_ctype('MG', 'manual')  # update if reclassified
# crest.save_cell_graph()

## Section 5: Duplicate Checking

During proofreading, the same physical neuron can sometimes be split across multiple segment IDs in the agglomeration, resulting in duplicate reconstructions. The duplicate check compares the base segments (root IDs) of all reconstructed cells to find overlaps between the current cell and any other cell in the dataset.

`get_base_segments_dict` builds a dictionary of `{cell_id: base_segment_set}` for all `.json` files in a directory. `check_duplicates` then compares the current cell's base segments against that dictionary and returns a dataframe of any overlapping cells.

*Tip: You do not need to re-run `get_base_segments_dict` every time. For each new crest instance, you can skip directly to the `check_duplicates` step.*

In [ ]:
# Build a dict mapping cell IDs to their base segment sets
base_segments_net = efish.get_base_segments_dict(Path(settings_dict['save_dir']))

In [ ]:
# Check if the current reconstruction shares base segments with any other cell
# crest.check_duplicates(base_segments_net)

## Section 6: Spine Density Annotation

Spine density was annotated by placing volume annotations at locations along the apical dendrite where spine density was visually assessed in the EM volume. These point-plus-radius annotations mark regions used to compute spine density estimates reported in Figure 2.

A `volume` annotation layer is added to the Neuroglancer viewer, and each annotation records a center coordinate (in Neuroglancer voxels) and radii defining the assessed region along the dendrite.

In [ ]:
# Add a volume annotation layer for spine density locations
# crest.add_endpoint_annotation_layers(['volume'], link=False)

In [ ]:
# Add a spine density annotation at a specific Neuroglancer location
# l = 5000  # distance along dendrite in nm
# crest.add_endpoint_annotation('spineD loc', to_vox=False, center=[17497, 6482, 2001], radii=[l/16, l/16, l/30])
# crest.save_cell_graph()